In [1]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample


import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# for CI testing
smoke_test = ('CI' in os.environ)
assert pyro.__version__.startswith('1.9.1')
pyro.set_rng_seed(1)


# Set matplotlib settings
%matplotlib inline
plt.style.use('default')


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class _BayesianLinearRegression(PyroModule):

    def __init__(self, in_features, out_features, mu=0., sigma=1.):
        super().__init__()
        self.linear = PyroModule[nn.Linear](in_features, out_features)
        self.linear.weight = PyroSample(dist.Normal(mu, sigma).expand([out_features, in_features]).to_event(1))
        self.linear.bias = PyroSample(dist.Normal(mu, sigma).expand([out_features]).to_event(1))
        
    def forward(self, x, y=None):
        sigma = pyro.sample("sigma", dist.Uniform(0., 10.))
        mean = self.linear(x).squeeze(-1)
        with pyro.plate("data", x.shape[0]):
            obs = pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean


class BayesianLinearRegression(BaseParameter):
    def __init__(self, in_features, out_features):
        self.model = _BayesianLinearRegression(in_features, out_features)
        self.guide = AutoDiagonalNormal(self.model)
        super().__init__()
        self.in_features =in_features
        self.out_features=out_features

    def fit(self, X, y):
        adam = pyro.optim.Adam({"lr": 0.03})
        svi = SVI(self.model, self.guide, adam, loss=Trace_ELBO())
        pyro.clear_param_store()
        for j in range(num_iterations):
            # calculate the loss and take a gradient step
            loss = svi.step(X, y)
            if j % 100 == 0:
                print("[iteration %04d] loss: %.4f" % (j + 1, loss / len(data)))

        self.guide.requires_grad_(False)


        for name, value in pyro.get_param_store().items():
            if name == "AutoDiagonalNormal.loc":
                means = pyro.param(name)
                means_np = means.detach().cpu().numpy()
            if name == "AutoDiagonalNormal.scale":
                sigmas = pyro.param(name)
                sigmas_np = sigmas.detach().cpu().numpy()
        self.means_ = means_np[:-1]
        self.sigmas_ = sigmas_np[:-1]
        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X_array = X.to_numpy(dtype=float)
            index = X.index
        else:
            X_array = np.asarray(X, dtype=float)
            index = pd.RangeIndex(len(X_array))

        if X_array.ndim == 1:
            X_array = X_array.reshape(1, -1)
            index = pd.RangeIndex(1)

        means = np.asarray(self.means_, dtype=float).reshape(-1)
        sigmas = np.asarray(self.sigmas_, dtype=float).reshape(-1)

        # intercept_mean + x1*mean1 + x2*mean2 + ...
        pred_mean = means[0] + X_array @ means[1:]

        # intercept_var + x1²*var1 + x2²*var2 + ...
        pred_variance = (
            sigmas[0] ** 2
            + (X_array**2) @ (sigmas[1:] ** 2)
        )

        pred_sigma = np.sqrt(pred_variance)

        return SkproNormal(
            mu=pred_mean.reshape(-1, 1),
            sigma=pred_sigma.reshape(-1, 1),
            index=index,
            columns=pd.Index(["y"]),
        )


In [3]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)
x_data, y_data = data[:, :-1], data[:, -1]

# Define loss and optimize
loss_fn = torch.nn.MSELoss(reduction='sum')
num_iterations = 1500 if not smoke_test else 2


In [4]:
linear_model = BayesianLinearRegression(3, 1)


In [5]:
linear_model.fit(x_data, y_data)


[iteration 0001] loss: 4.4480
[iteration 0101] loss: 3.0572
[iteration 0201] loss: 2.5214
[iteration 0301] loss: 1.9759
[iteration 0401] loss: 1.6874
[iteration 0501] loss: 1.6745
[iteration 0601] loss: 1.7007
[iteration 0701] loss: 1.6924
[iteration 0801] loss: 1.6943
[iteration 0901] loss: 1.6901
[iteration 1001] loss: 1.6891
[iteration 1101] loss: 1.6780
[iteration 1201] loss: 1.6973
[iteration 1301] loss: 1.6871
[iteration 1401] loss: 1.6849


BayesianLinearRegression(in_features=3, out_features=1)

In [6]:
import pandas as pd

X_df = pd.DataFrame(
    x_data[:5].detach().cpu().numpy(),
    columns=["x1", "x2", "x3"],
)

pred_dist = linear_model.predict_proba(X_df)


In [7]:
pred_dist


Normal(columns=Index(['y'], dtype='object'),
       index=RangeIndex(start=0, stop=5, step=1),
       mu=array([[-3.70874097],
       [-2.48917309],
       [-2.28978506],
       [-2.29023514],
       [-2.43373751]]),
       sigma=array([[0.18086412],
       [0.14888965],
       [0.06728976],
       [0.06739855],
       [0.12255866]]))

In [8]:
pred_dist.mu.shape


(5, 1)